# Part 2 — one controlled experiment: shared local temporal tokens

**Question:** does a temporal structure prior help attention generalize across people, rather than merely memorize 360 EEG trials?

**One bet:** eight 200 ms temporal filters shared across all electrodes, at 100 ms stride, produce 29 overlapping tokens plus CLS. The transformer backbone stays unchanged. Proposed model and vanilla each have **135,746 parameters**. Attention score entries fall **258×**, but measured speedup need not match that ratio.

**Four configurations:** Part-1-config EEGNet, unchanged vanilla transformer, proposed local-token model, and an equally sized scattered-time control. The control preserves the token count and learned weights but removes ordered contiguous grouping.

A well-controlled negative finding can satisfy the assessment. Higher accuracy is evidence, not the grading objective. Reduced variability from predicting only one class is not success.

The baseline test scores are already known, so this is an **exploratory** follow-up. Read the locked plan before running. No hyperparameter sweep or winning-seed selection is performed.

Upload only `part2_transformer` (including `dataset.zip` and all new Python files) to Drive. Select **Runtime → Change runtime type → T4 GPU**, set the folder path below, and run all cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json
from datetime import datetime, timezone

# EDIT this direct Part 2 folder path if needed.
DRIVE_PART2 = Path('/content/drive/MyDrive/Basil AI Intern OA/part2_transformer')
if not (DRIVE_PART2 / 'experiment.py').is_file():
    raise FileNotFoundError(f'Upload the updated Part 2 folder and set DRIVE_PART2. Looked in {DRIVE_PART2}')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
LOCAL_PART2 = Path('/content') / f'basil_study_{stamp}'
LOCAL_PART2.mkdir()
for pattern in ['*.py', '*.md', 'requirements*.txt']:
    for source in DRIVE_PART2.glob(pattern):
        shutil.copy2(source, LOCAL_PART2 / source.name)
if (DRIVE_PART2 / 'dataset.zip').is_file():
    shutil.copy2(DRIVE_PART2 / 'dataset.zip', LOCAL_PART2 / 'dataset.zip')
elif (DRIVE_PART2 / '.cache/mne').is_dir():
    shutil.copytree(DRIVE_PART2 / '.cache/mne', LOCAL_PART2 / '.cache/mne')
else:
    print('No archive found: the preparation step will download the recordings.')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
DRIVE_OUTPUTS = DRIVE_PART2 / 'artifacts/controlled_experiment'
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Local:', LOCAL_PART2)
print('Drive:', DRIVE_OUTPUTS)


## Read the hypothesis and endpoints before training
The plan is copied into each output folder and its SHA-256 is recorded. Full descriptive support requires better mean balanced accuracy than vanilla **and** scattered control, seed SD no larger than vanilla, mean above chance, and no collapsed local-model seed. These are pilot criteria, not a statistical significance test. Partial/negative results remain reportable.

In [ ]:
from IPython.display import display, Markdown
import hashlib
plan = LOCAL_PART2 / 'EXPERIMENT_PLAN.md'
print('Plan SHA-256:', hashlib.sha256(plan.read_bytes()).hexdigest())
display(Markdown(plan.read_text()))


## Dependencies and GPU
Retain Colab's CUDA PyTorch. Strict deterministic math attention, FP16 gradient scaling, local data, matched minibatch orders, and exact repeated-seed checks are built into the runner. Exactness applies to this GPU/software stack.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LOCAL_PART2 / 'requirements-colab.txt')], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then reconnect and run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('Capability:', torch.cuda.get_device_capability(0))
subprocess.run(['nvidia-smi'], check=True)


## Verify and preprocess recordings
The preparation step extracts all 30 files and validates their checksums. The study reloads unstandardized epochs and independently refits scaling: subjects 1–6 for development, subjects 1–8 for final evaluation. It does **not** reuse a scaler fitted on development evaluation subjects.

In [ ]:
subprocess.run([sys.executable, '-B', '-u', 'prepare_data.py'], cwd=LOCAL_PART2, check=True)
import json
print(json.dumps(json.loads((LOCAL_PART2 / 'artifacts/data_summary.json').read_text()), indent=2))


## Run the locked experiment

- **Development:** train 1–6, diagnose 7–8. Plot evaluation loss every epoch, with no checkpoint selection.
- **Final:** retrain 1–8, evaluate 9–10 only after the fixed final epoch.
- **All four models:** seeds 0–4, 40 epochs, AdamW lr 0.001, effective batch 64. Same per-seed minibatch order across models.
- Repeat seed 0 for every model in each stage and require exact weights, predictions and losses.

Both stages run with the same settings regardless of the development outcome. No model/epoch is selected using test performance. There are 40 scientific runs plus eight repeat checks; the short-token models are inexpensive, and actual GPU times are recorded. Completed runs are backed up as they finish. Hardware/package versions are recorded; earlier three-seed scores are not mixed into the new five-seed summaries.

In [ ]:
EPOCHS = 40
SEEDS = [0, 1, 2, 3, 4]
STAGES = ['dev', 'final']
VERIFY_REPEATS = True

command = [sys.executable, '-B', '-u', 'experiment.py', '--device', 'cuda',
           '--stages', *STAGES, '--epochs', str(EPOCHS), '--seeds', *map(str, SEEDS),
           '--sync-dir', str(DRIVE_OUTPUTS)]
if not VERIFY_REPEATS:
    command.append('--no-verify-repeat')
import signal
log_path = LOCAL_PART2 / 'artifacts/experiment_launcher.log'
process = None
try:
    with log_path.open('w') as logfile:
        process = subprocess.Popen(command, cwd=LOCAL_PART2, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            logfile.write(line)
            logfile.flush()
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Study failed (exit {returncode}). Inspect the log and saved metadata.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=60)
        except subprocess.TimeoutExpired:
            process.terminate()
    raise
finally:
    if log_path.exists():
        shutil.copy2(log_path, DRIVE_OUTPUTS / f'launcher_{stamp}.log')


## Inspect the outcome, including failed criteria
Read the development and final results separately. A local-vs-scattered difference is stronger evidence for ordered local tokenization than a local-vs-vanilla difference alone. The experiment report gives the caveats; no automatic physiological conclusion is made.

In [ ]:
from IPython.display import Image
runs = sorted((LOCAL_PART2 / 'artifacts').glob('study_*'))
completed = [p for p in runs if p.is_dir() and not p.name.startswith('study_smoke_')
             and (p / 'metadata.json').is_file()
             and json.loads((p / 'metadata.json').read_text()).get('status') == 'complete']
if not completed:
    raise RuntimeError('No completed real study. Check the training log.')
RUN_DIR = completed[-1]
display(Markdown((RUN_DIR / 'EXPERIMENT_REPORT.md').read_text()))
for stage in STAGES:
    print(stage.upper())
    display(Image(filename=str(RUN_DIR / stage / 'experiment_dashboard.png')))
if (RUN_DIR / 'ANSWER_DRAFT.md').exists():
    display(Markdown((RUN_DIR / 'ANSWER_DRAFT.md').read_text()))
print('Saved on Drive:', DRIVE_OUTPUTS / RUN_DIR.name)


## Export results
The ZIP includes the hypothesis snapshot, source hashes, hardware/software metadata, all seed results, paired control differences, exact-repeat checks, figures, checkpoints, and a measured write-up draft. Inspect and explain the actual outcome before using the draft in `ANSWERS.md`.

In [ ]:
archive = Path(shutil.make_archive(str(LOCAL_PART2 / RUN_DIR.name), 'zip', root_dir=RUN_DIR))
DRIVE_EXPORT = DRIVE_OUTPUTS / archive.name
shutil.copy2(archive, DRIVE_EXPORT)
print('Export ZIP:', DRIVE_EXPORT)
DOWNLOAD_TO_LAPTOP = False
if DOWNLOAD_TO_LAPTOP:
    from google.colab import files
    files.download(str(archive))
